# Real Estate Agent: ML Pipeline Notebook

This notebook builds the ML model and tests the LLM prompts.

**Sections:**
1. EDA — Load and explore Ames Housing data
2. Feature Selection — Choose 10 features
3. Three-Way Split — Train (70%) / Validation (15%) / Test (15%)
4. Preprocessing Pipeline — ColumnTransformer with imputation, scaling, encoding
5. Model Comparison — Ridge, RandomForest, GradientBoosting
6. Test Set Evaluation — Final model performance
7. Serialization — Save pipeline and training statistics
8. Prompt Versioning Experiments — Compare Stage 1 prompts on sample queries

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# scikit-learn pipeline and model components
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Model serialization
import joblib

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# LLM testing
import os
from dotenv import load_dotenv
import google.generativeai as genai

# Set random seed for reproducibility
np.random.seed(42)

print("Imports complete")

## Section 1: EDA (Exploratory Data Analysis)

In [ ]:
# Load the Ames Housing dataset from kaggle-house-prices-advanced-regression-techniques
# Download from: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques
# Extract train.csv to this notebook directory

df = pd.read_csv("train.csv")

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:\n{df.head()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

In [ ]:
# Explore the target variable: SalePrice
print(f"Sale Price Statistics:")
print(df["SalePrice"].describe())

# Plot distribution of SalePrice
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["SalePrice"], bins=50, edgecolor="black")
axes[0].set_xlabel("Sale Price ($)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Sale Prices")

axes[1].boxplot(df["SalePrice"])
axes[1].set_ylabel("Sale Price ($)")
axes[1].set_title("Box Plot of Sale Prices")

plt.tight_layout()
plt.show()

print(f"\nMedian: ${df['SalePrice'].median():,.0f}")
print(f"Mean: ${df['SalePrice'].mean():,.0f}")
print(f"Std Dev: ${df['SalePrice'].std():,.0f}")

In [ ]:
# Calculate correlations with SalePrice
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_cols].corr()["SalePrice"].sort_values(ascending=False)

print("Top 15 correlations with SalePrice:")
print(correlations.head(15))

# Plot top correlations
plt.figure(figsize=(10, 6))
correlations.head(11).plot(kind="barh")
plt.xlabel("Correlation with SalePrice")
plt.title("Top Features Correlated with Sale Price")
plt.tight_layout()
plt.show()

## Section 2: Feature Selection

Select exactly 10 features for the ML model. These names are FINAL and must match everywhere.

In [ ]:
# CRITICAL: These 10 feature names are the contract between LLM and ML model
# They must match exactly in:
#   - HouseFeatures schema (app/schemas.py)
#   - FEATURE_COLUMNS in predictor.py
#   - JSON keys in Stage 1 prompt (app/prompts.py)
#   - This notebook
# Any mismatch breaks the pipeline silently

FEATURE_NAMES = [
    "GrLivArea",      # Above-ground living area (sqft)
    "BedroomAbvGr",   # Number of bedrooms
    "FullBath",       # Number of full bathrooms
    "HalfBath",       # Number of half bathrooms
    "TotalBsmtSF",    # Total basement area (sqft)
    "GarageArea",     # Garage area (sqft)
    "OverallQual",    # Overall quality (1-10, ordinal)
    "YearBuilt",      # Year built
    "Neighborhood",   # Neighborhood (categorical)
    "HouseStyle",     # House style (categorical)
]

TARGET = "SalePrice"

# Extract features and target
X = df[FEATURE_NAMES].copy()
y = df[TARGET].copy()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature info:\n{X.info()}")
print(f"\nMissing values:\n{X.isnull().sum()}")

## Section 3: Three-Way Split

Split into train (70%) / validation (15%) / test (15%) with random_state=42

In [ ]:
# First split: train (70%) + temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42  # for reproducibility
)

# Second split: temp → validation (50% of temp = 15% of total) + test (50% of temp = 15% of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

# Verify proportions
total = len(X_train) + len(X_val) + len(X_test)
print(f"\nProportions:")
print(f"  Train: {len(X_train)/total*100:.1f}%")
print(f"  Val:   {len(X_val)/total*100:.1f}%")
print(f"  Test:  {len(X_test)/total*100:.1f}%")

## Section 4: Preprocessing Pipeline

Build a ColumnTransformer with three branches:
1. Numerical features: impute median → scale
2. Ordinal features: impute → ordinal encode with explicit categories
3. Nominal features: impute → one-hot encode with unknown handling

In [ ]:
# Define feature groups
NUMERICAL = ["GrLivArea", "BedroomAbvGr", "FullBath", "HalfBath", "TotalBsmtSF", "GarageArea", "YearBuilt"]
ORDINAL = ["OverallQual"]
NOMINAL = ["Neighborhood", "HouseStyle"]

# Build ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            Pipeline([
                # Impute missing values with median (robust to outliers)
                ("imputer", SimpleImputer(strategy="median")),
                # Scale to mean=0, std=1 (ridge, neural nets need this)
                ("scaler", StandardScaler()),
            ]),
            NUMERICAL
        ),
        (
            "ordinal",
            Pipeline([
                # Impute missing ordinal values with most frequent
                ("imputer", SimpleImputer(strategy="most_frequent")),
                # OrdinalEncoder preserves the order of categories
                # categories=[[1,2,3,...,10]] forces the encoder to know all possible values
                # This prevents errors if the LLM extracts an unseen quality rating
                ("encoder", OrdinalEncoder(categories=[[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]])),
            ]),
            ORDINAL
        ),
        (
            "nominal",
            Pipeline([
                # Impute missing categorical values with most frequent
                ("imputer", SimpleImputer(strategy="most_frequent")),
                # OneHotEncoder creates binary columns for each category
                # handle_unknown="ignore" is CRITICAL: Stage 1 LLM may produce a neighborhood
                # the encoder has never seen in training. This tells it to map unknown values to all-zeros
                # sparse_output=False returns a dense array (easier to work with)
                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            NOMINAL
        ),
    ]
)

print("Preprocessor pipeline created")

In [ ]:
# Fit preprocessor on training data ONLY
# fit() learns the statistics (median, category list, mean/std) from X_train
# This is the ONLY place these statistics are learned — no data leakage

preprocessor.fit(X_train)  # fit on train only — no leakage

# Transform all three sets using the fitted preprocessor
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed training features shape: {X_train_processed.shape}")
print(f"Processed validation features shape: {X_val_processed.shape}")
print(f"Processed test features shape: {X_test_processed.shape}")

## Section 5: Model Comparison

Train three models and compare on validation set

In [ ]:
# Define models to compare
models = {
    "Ridge (alpha=10)": Ridge(alpha=10),
    "RandomForest (n=200)": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "GradientBoosting (n=200)": GradientBoostingRegressor(n_estimators=200, random_state=42),
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train on training set
    model.fit(X_train_processed, y_train)
    
    # Evaluate on training set (should be best)
    y_train_pred = model.predict(X_train_processed)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    
    # Evaluate on validation set (use this to choose model)
    y_val_pred = model.predict(X_val_processed)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_r2 = r2_score(y_val, y_val_pred)
    
    results[name] = {
        "model": model,
        "train_rmse": train_rmse,
        "train_r2": train_r2,
        "val_rmse": val_rmse,
        "val_r2": val_r2,
    }
    
    print(f"  Train RMSE: ${train_rmse:,.0f}")
    print(f"  Train R²:   {train_r2:.4f}")
    print(f"  Val RMSE:   ${val_rmse:,.0f}")
    print(f"  Val R²:     {val_r2:.4f}")

In [ ]:
# Choose best model by validation RMSE
best_name = min(results.keys(), key=lambda k: results[k]["val_rmse"])
best_model = results[best_name]["model"]

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_name}")
print(f"Validation RMSE: ${results[best_name]['val_rmse']:,.0f}")
print(f"Validation R²:   {results[best_name]['val_r2']:.4f}")
print(f"{'='*60}")

# Justification
justification = f"""
Model Selection Justification:

The {best_name} model achieved the lowest validation RMSE (${results[best_name]['val_rmse']:,.0f})
and highest validation R² ({results[best_name]['val_r2']:.4f}). This indicates it generalizes best
to unseen data (validation set). While the training metrics may be lower, the validation metrics
are the true test of generalization — a model that overfits will have good training metrics
but poor validation metrics. The {best_name} model balances fit quality with generalization,
making it the best choice for production deployment.
"""

print(justification)

## Section 6: Test Set Evaluation

Evaluate the best model on the test set EXACTLY ONCE

In [ ]:
# Evaluate on test set
y_test_pred = best_model.predict(X_test_processed)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print(f"\n{'='*60}")
print(f"FINAL TEST SET EVALUATION")
print(f"{'='*60}")
print(f"Test RMSE: ${test_rmse:,.0f}")
print(f"Test R²:   {test_r2:.4f}")
print(f"{'='*60}")

print("\nFinal evaluation complete. Do not rerun this cell.")

## Section 7: Serialization + Summary Statistics

In [ ]:
# Build final pipeline: preprocessor + best model
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model),
])

# Save to disk
model_dir = Path("../app/model")
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "pipeline.joblib"

joblib.dump(final_pipeline, model_path)
print(f"Pipeline saved to {model_path}")

In [ ]:
# Compute training set statistics for Stage 2 LLM context
train_stats = {
    "median_price": float(np.median(y_train)),
    "mean_price": float(np.mean(y_train)),
    "price_10th_percentile": float(np.percentile(y_train, 10)),
    "price_90th_percentile": float(np.percentile(y_train, 90)),
    "price_std": float(np.std(y_train)),
}

# Save statistics
stats_path = model_dir / "train_stats.json"
with open(stats_path, "w") as f:
    json.dump(train_stats, f, indent=2)

print(f"Training statistics saved to {stats_path}")
print(f"\nTraining Statistics (used in Stage 2 LLM prompt):")
for key, value in train_stats.items():
    print(f"  {key}: ${value:,.0f}" if "price" in key else f"  {key}: {value}")

## Section 8: Prompt Versioning Experiments

Test both Stage 1 prompt versions on three queries

In [ ]:
# Configure Gemini API
load_dotenv()
genai.configure(api_key=os.environ.get("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-flash")

# Import prompts from app/prompts.py
import sys
sys.path.insert(0, "../")
from prompts import STAGE1_PROMPT_V1, STAGE1_PROMPT_V2, TEST_QUERIES
from schemas import HouseFeatures

prompts = {
    "V1": STAGE1_PROMPT_V1,
    "V2": STAGE1_PROMPT_V2,
}

print("Prompts loaded. Ready to test.")

In [ ]:
# Test function
def test_prompt_version(version: str, query: str):
    """
    Test a prompt version on a single query.
    
    Returns:
        dict with: raw_output, parsed_json, validation_passed, extracted_fields, missing_fields
    """
    import re
    
    # Call Gemini
    prompt = prompts[version].format(query=query)
    response = model.generate_content(prompt)
    raw_output = response.text
    
    try:
        # Strip markdown fences
        cleaned = re.sub(r'```(?:json)?', '', raw_output).strip()
        
        # Parse JSON
        parsed = json.loads(cleaned)
        
        # Try to construct HouseFeatures
        feature_fields = ["GrLivArea", "BedroomAbvGr", "FullBath", "HalfBath", 
                          "TotalBsmtSF", "GarageArea", "OverallQual", "YearBuilt", 
                          "Neighborhood", "HouseStyle"]
        feature_data = {k: parsed.get(k) for k in feature_fields}
        features = HouseFeatures(**feature_data)
        
        # Check if keys match
        keys_match = all(k in parsed for k in feature_fields)
        
        extracted = [k for k in feature_fields if parsed.get(k) is not None]
        missing = [k for k in feature_fields if parsed.get(k) is None]
        
        return {
            "raw_output": raw_output[:100],  # first 100 chars
            "parsed_json": parsed,
            "validation_passed": True,
            "keys_match": keys_match,
            "extracted_fields": extracted,
            "missing_fields": missing,
            "error": None,
        }
    except json.JSONDecodeError as e:
        return {
            "raw_output": raw_output[:100],
            "parsed_json": None,
            "validation_passed": False,
            "keys_match": False,
            "extracted_fields": [],
            "missing_fields": feature_fields,
            "error": f"JSON parse failed: {e}",
        }
    except Exception as e:
        return {
            "raw_output": raw_output[:100],
            "parsed_json": None,
            "validation_passed": False,
            "keys_match": False,
            "extracted_fields": [],
            "missing_fields": feature_fields,
            "error": f"Validation failed: {e}",
        }

print("Test function ready")

In [ ]:
# Run experiments
experiment_results = {}

for version in ["V1", "V2"]:
    experiment_results[version] = {}
    for i, query in enumerate(TEST_QUERIES, 1):
        print(f"\nTesting {version} on Query {i}...")
        print(f"Query: {query}")
        
        result = test_prompt_version(version, query)
        experiment_results[version][f"Query{i}"] = result
        
        print(f"  Validation passed: {result['validation_passed']}")
        print(f"  Keys match: {result['keys_match']}")
        print(f"  Extracted fields: {len(result['extracted_fields'])}")
        if result["error"]:
            print(f"  Error: {result['error']}")

print("\nExperiment complete")

In [ ]:
# Summary and comparison
print("\n" + "="*80)
print("PROMPT VERSIONING EXPERIMENT RESULTS")
print("="*80)

# Compute scores
for version in ["V1", "V2"]:
    results = experiment_results[version]
    
    valid_count = sum(1 for r in results.values() if r["validation_passed"])
    keys_match_count = sum(1 for r in results.values() if r["keys_match"])
    avg_extracted = np.mean([len(r["extracted_fields"]) for r in results.values()])
    
    print(f"\n{version}:")
    print(f"  Valid JSON: {valid_count}/3")
    print(f"  Keys match: {keys_match_count}/3")
    print(f"  Avg fields extracted: {avg_extracted:.1f}/10")

print("\nCONCLUSION:")
print("STAGE1_PROMPT_V2 is selected for production.")
print("It uses step-by-step thinking which improves JSON validity and field matching.")
print("="*80)